# FrameGuard: mixed-dataset training run on a Kaggle GPU

Trains FrameGuard from scratch (EfficientNet-B0 spatial CNN with augmentation, then LSTM) on a pooled
dataset: FaceForensics++ C23 (Deepfakes, Face2Face, FaceSwap, NeuralTextures), Celeb-DF v2 and DFDC.
Each is split 70/15/15 by connected source/identity groups, so no person or source video crosses
splits. FF++ FaceShifter is kept out of training and tested as an **unseen manipulation method**.

## Before you run

1. **Add data** (right panel → *Add Input*), all three datasets:
   - `xdxd003/ff-c23` (FaceForensics++ C23)
   - `reubensuju/celeb-df-v2` (Celeb-DF v2)
   - `pranay22077/dfdc-10` (DFDC training parts 00–09, ~104 GB)
2. **Settings → Accelerator → GPU T4**.
3. **Settings → Internet → On** (the notebook clones the code and installs MediaPipe).
   Kaggle requires a phone-verified account for GPU and internet.
4. To run in the background with the browser closed: **Save Version → Save & Run All (Commit)**.
   Results are kept in the version's **Output** tab. A run has a 12-hour limit.

Expected time: roughly 6–8 hours (an estimate). Face preparation prints its speed and an ETA every
25 videos. If the preparation ETA alone goes past ~7 hours, stop the run and lower `DFDC_REALS`.

In [ ]:
# Settings.
import os
EPOCHS = 5              # per training stage (spatial CNN and LSTM)
BATCH_SIZE = 32
FRAMES_PER_VIDEO = 16   # spatial crops drawn per training video per epoch
WORKERS = 4             # DataLoader worker processes
MAX_SAMPLES = 64        # frames sampled per video (8 fps, so ~8 s per video)
PREP_WORKERS = os.cpu_count()  # parallel face-preparation processes
CELEB_FAKES = 2500      # Celeb-DF fakes sampled (all Celeb-DF real videos are used)
DFDC_REALS = 3000       # DFDC real videos sampled, each paired with one of its fakes
HOLDOUT = "FaceShifter" # FF++ method kept out of training and tested as unseen ("" = none)
REPO = "https://github.com/khakash-060206/FrameGuard.git"

In [ ]:
import os, shutil, subprocess, sys
print(sys.version)
subprocess.run(["nvidia-smi"], check=False)
for path in ("/kaggle/working", "/kaggle/temp", "/tmp"):
    if os.path.exists(path):
        print(f"{path}: {shutil.disk_usage(path).free / 1024**3:.0f} GB free")

## 1. Get the code and install dependencies

In [ ]:
PROJECT = "/kaggle/working/FrameGuard"
if not os.path.exists(PROJECT):
    subprocess.run(["git", "clone", "--depth", "1", REPO, PROJECT], check=True)
else:
    subprocess.run(["git", "-C", PROJECT, "pull", "--ff-only"], check=True)
os.chdir(PROJECT)
subprocess.run(["git", "log", "--oneline", "-1"], check=True)

In [ ]:
# Kaggle already has PyTorch, torchvision, NumPy and scikit-learn. MediaPipe is the only missing package.
# All OpenCV variants share one cv2 folder, so removing one breaks the others. Remove every variant,
# then install exactly one (opencv-contrib, which MediaPipe requires) fresh.
OPENCV = ["opencv-python", "opencv-python-headless", "opencv-contrib-python", "opencv-contrib-python-headless"]
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *OPENCV], check=False)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--force-reinstall", "--no-deps",
                "opencv-contrib-python==4.14.0.94"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "mediapipe==0.10.35"], check=True)

In [ ]:
# Verify the stack in a fresh process (the kernel may still hold the old cv2).
check = '''
import torch, torchvision, cv2, mediapipe, numpy, sklearn
assert hasattr(cv2, "VideoCapture"), f"Broken OpenCV install at {cv2.__path__}; rerun the install cell"
print("torch", torch.__version__, "| torchvision", torchvision.__version__, "| opencv", cv2.__version__,
      "| mediapipe", mediapipe.__version__, "| numpy", numpy.__version__, "| sklearn", sklearn.__version__)
assert torch.cuda.is_available(), "No GPU: set Settings -> Accelerator -> GPU"
print("GPU:", torch.cuda.get_device_name(0))
'''
subprocess.run([sys.executable, "-c", check], check=True)

## 2. Find the datasets and choose where to store face crops

In [ ]:
from pathlib import Path

def find(marker, kind):
    for path in sorted(Path("/kaggle/input").rglob(marker)):
        if (kind == "dir" and path.is_dir()) or (kind == "file" and path.is_file()):
            return path.parent
    raise FileNotFoundError(f"{marker} not found under /kaggle/input. Add the dataset (see the top of the notebook).")

FFPP = find("FaceForensics++_C23", "dir")
CELEB = find("List_of_testing_videos.txt", "file")
parts = [m.parent for m in Path("/kaggle/input").rglob("metadata.json") if m.parent.name.startswith("dfdc_train_part")]
if not parts:
    raise FileNotFoundError("DFDC metadata.json files not found. Add the pranay22077/dfdc-10 dataset (see the top).")
DFDC = Path(os.path.commonpath([str(p) for p in parts]))
print("FF++ :", FFPP)
print("Celeb:", CELEB)
print(f"DFDC : {DFDC} ({len(parts)} parts with metadata.json)")

# Face crops (~150 GB at 64 frames/video for ~15k videos) must not go in /kaggle/working (20 GB output limit).
candidates = [p for p in ("/kaggle/temp", "/tmp") if os.path.isdir(p)]
DATA_DIR = max(candidates, key=lambda p: shutil.disk_usage(p).free) + "/frameguard-data"
free_gb = shutil.disk_usage(os.path.dirname(DATA_DIR)).free / 1024**3
needed_gb = 15500 * MAX_SAMPLES * 150 / 1024**2 + 30
if free_gb < needed_gb:
    raise RuntimeError(f"Only {free_gb:.0f} GB free; about {needed_gb:.0f} GB is needed. Lower MAX_SAMPLES or DFDC_REALS.")
print(f"Data dir: {DATA_DIR} ({free_gb:.0f} GB free, ~{needed_gb:.0f} GB needed) | frames per video: {MAX_SAMPLES} | "
      f"prep workers: {PREP_WORKERS}")

## 3. Run the full experiment

Import → pooled split → face crops → spatial CNN → feature cache → LSTM → evaluation. Progress streams below and is also written to `runs/experiment-status.json`.

In [ ]:
command = [sys.executable, "-u", "scripts/run_experiment.py",
           "--ffpp", str(FFPP), "--celeb", str(CELEB), "--dfdc", str(DFDC), "--data-dir", DATA_DIR,
           "--device", "cuda", "--epochs", str(EPOCHS), "--batch-size", str(BATCH_SIZE),
           "--max-samples", str(MAX_SAMPLES), "--prep-workers", str(PREP_WORKERS), "--workers", str(WORKERS),
           "--celeb-fakes", str(CELEB_FAKES), "--dfdc-reals", str(DFDC_REALS), "--holdout", HOLDOUT,
           "--augment", "on", "--amp", "on", "--resume"]
if FRAMES_PER_VIDEO:
    command += ["--frames-per-video", str(FRAMES_PER_VIDEO)]
print(" ".join(command), flush=True)

with open("runs-console.log", "w") as log:
    process = subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        log.write(line)
        # MediaPipe prints harmless telemetry upload errors; keep them out of the notebook view.
        if "clearcut" not in line and "Source Location Trace" not in line and "portable_clearcut" not in line:
            print(line, end="", flush=True)
    code_ = process.wait()
if code_ != 0:
    raise RuntimeError(f"Experiment failed (exit code {code_}). See runs/experiment-status.json and runs-console.log.")

## 4. Results

In [ ]:
import json

def show(path):
    report = json.loads(Path(path).read_text())
    print(f"\n=== {path} (split: {report['split']}, unit: {report['unit']}) ===")
    print(f"{'model':<10}{'group':<34}{'videos':>7}{'ROC-AUC':>9}{'F1':>7}{'Prec':>7}{'Recall':>8}{'FPR':>7}")
    for model, domains in report["metrics"].items():
        for domain, m in domains.items():
            print(f"{model:<10}{domain:<34}{m['count']:>7}{m['roc_auc']:>9.3f}{m['f1']:>7.3f}"
                  f"{m['precision']:>7.3f}{m['recall']:>8.3f}{m['false_positive_rate']:>7.3f}")
    print(f"Unassessed videos: {len(report['unassessed'])}")

for stage in ("spatial", "temporal"):
    history = json.loads(Path(f"models/{stage}.history.json").read_text())
    best = max(history, key=lambda e: e["validation"]["roc_auc"])
    print(f"{stage}: best validation ROC-AUC {best['validation']['roc_auc']:.3f} at epoch {best['epoch']}/{len(history)}")
show("runs/mixed-test.json")
print(json.dumps(json.loads(Path(DATA_DIR + "/imported/inventory.json").read_text())["by_dataset"], indent=1))

In [ ]:
# Bundle everything needed to view the results locally into one downloadable file (Output tab).
import zipfile
# The run's dataset inventory (split sizes, prepared counts) lets the local dashboard describe this model's data.
inventory = json.loads(Path(DATA_DIR + "/imported/inventory.json").read_text())
prepared = json.loads(Path(DATA_DIR + "/crops/index.json").read_text())
inventory.update(prepared_videos=len(prepared["videos"]), rejected_videos=len(prepared["rejected"]))
Path("runs/inventory.json").write_text(json.dumps(inventory, indent=2))
bundle = Path("/kaggle/working/frameguard-results.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as archive:
    for pattern in ("models/*.pt", "models/*.history.json", "runs/*.json", "runs-console.log"):
        for file in Path(".").glob(pattern):
            archive.write(file, file.as_posix())
print(bundle, f"{bundle.stat().st_size / 1024**2:.1f} MB")

## Viewing results on your own computer

Download `frameguard-results.zip` from the version's **Output** tab, unzip it into your local
FrameGuard folder (it contains `models/` and `runs/`), then run:

```
.\.venv\Scripts\python.exe -m frameguard dashboard
```

and open http://127.0.0.1:8765.